# 🧩 Dynamic Programming: Deduplicating the Recursion Tree

**DP isn't magic — it's memoization. You already understand recursion trees. DP just removes the duplicate nodes.**

---

## 1. Concept Intuition

Recall from the combinatorics module: recursion builds a **tree** of subproblems.

Sometimes, the same subproblem appears **many times** in different branches. Computing it over and over is wasted work.

**Dynamic Programming = recursion + cache.**

The idea: when you solve a subproblem, **store the answer**. Next time you need it, look it up instead of recomputing.

### The two flavors
- **Top-down (memoization)**: recursive, but cache results. Natural to write.
- **Bottom-up (tabulation)**: fill a table iteratively from small to large. Often more efficient.

Both produce the same answers. The difference is style, not substance.

## ⚠️ Common Wrong Intuition
*(Describe the wrong way most people first think about this. Show why it breaks, and explain how the correct intuition rectifies it.)*

## 2. Visual Representation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Fibonacci: the poster child of overlapping subproblems
# Count how many times each fib(k) is needed in the naive recursion tree
call_counts = {}
def fib_count(n):
    call_counts[n] = call_counts.get(n, 0) + 1
    if n <= 1: return n
    return fib_count(n-1) + fib_count(n-2)

call_counts.clear()
fib_count(12)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# How many times each subproblem is computed
ks = sorted(call_counts.keys())
counts = [call_counts[k] for k in ks]
colors = ['#f43f5e' if c > 1 else '#10b981' for c in counts]
axes[0].bar(ks, counts, color=colors, alpha=0.8)
axes[0].set_xlabel('Subproblem fib(k)')
axes[0].set_ylabel('Times computed')
axes[0].set_title('Naive recursion: massive duplication!', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# With DP: each computed exactly once
dp_counts = [1] * len(ks)
axes[1].bar(ks, dp_counts, color='#10b981', alpha=0.8)
axes[1].set_xlabel('Subproblem fib(k)')
axes[1].set_ylabel('Times computed')
axes[1].set_title('With memoization: each computed ONCE', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

total_naive = sum(counts)
total_dp = sum(dp_counts)
plt.suptitle(f'Naive: {total_naive} calls  →  DP: {total_dp} calls  ({total_naive/total_dp:.0f}× savings)', 
            fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

## 3. Mathematical Formulation

### Optimal substructure
A problem has **optimal substructure** if the optimal solution contains optimal solutions to subproblems.

### Overlapping subproblems
A problem has **overlapping subproblems** if the recursion revisits the same states.

**DP applies when BOTH conditions are present.**

### Fibonacci recurrence
$$F(n) = F(n-1) + F(n-2), \quad F(0)=0, F(1)=1$$

- Naive: $O(2^n)$ (exponential tree)
- With DP: $O(n)$ (each of $n$ subproblems computed once)

## 4. Code Implementation

### Three approaches to Fibonacci

In [ ]:
import time

# 1. Naive recursive: O(2^n)
def fib_naive(n):
    if n <= 1: return n
    return fib_naive(n-1) + fib_naive(n-2)

# 2. Top-down DP (memoization): O(n)
def fib_memo(n, cache={}):
    if n in cache: return cache[n]
    if n <= 1: return n
    cache[n] = fib_memo(n-1, cache) + fib_memo(n-2, cache)
    return cache[n]

# 3. Bottom-up DP (tabulation): O(n)
def fib_tab(n):
    if n <= 1: return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i-1] + dp[i-2]
    return dp[n]

# Benchmark
sizes = list(range(5, 33))
naive_times = []
for n in sizes:
    if n <= 30:
        start = time.perf_counter()
        fib_naive(n)
        naive_times.append(time.perf_counter() - start)
    else:
        naive_times.append(None)

memo_times = []
for n in sizes:
    cache = {}
    start = time.perf_counter()
    fib_memo(n, cache)
    memo_times.append(time.perf_counter() - start)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sizes[:len(naive_times)], [t*1e6 if t else None for t in naive_times], 
        'o-', color='#f43f5e', linewidth=2, label='Naive O(2ⁿ)')
ax.plot(sizes, [t*1e6 for t in memo_times], 
        's-', color='#10b981', linewidth=2, label='Memoized O(n)')
ax.set_xlabel('n'); ax.set_ylabel('Time (μs)')
ax.set_title('The DP Speedup: Exponential → Linear', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Classic DP: coin change problem
# Minimum coins to make amount, given coin denominations

def coin_change(coins, amount):
    """Return minimum coins needed, and the coins used."""
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    parent = [-1] * (amount + 1)
    
    for a in range(1, amount + 1):
        for coin in coins:
            if coin <= a and dp[a - coin] + 1 < dp[a]:
                dp[a] = dp[a - coin] + 1
                parent[a] = coin
    
    # Reconstruct solution
    if dp[amount] == float('inf'):
        return -1, []
    
    used = []
    a = amount
    while a > 0:
        used.append(parent[a])
        a -= parent[a]
    return dp[amount], used

coins = [1, 5, 10, 25]
amount = 67
n_coins, used = coin_change(coins, amount)

# Visualize the DP table
dp = [float('inf')] * (amount + 1)
dp[0] = 0
for a in range(1, amount + 1):
    for c in coins:
        if c <= a: dp[a] = min(dp[a], dp[a-c] + 1)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(range(amount+1), dp, color='#6366f1', linewidth=1.5)
ax.fill_between(range(amount+1), dp, alpha=0.1, color='#6366f1')
for c in coins:
    ax.axvline(c, color='#f97316', linestyle=':', alpha=0.5)
ax.set_xlabel('Amount'); ax.set_ylabel('Min coins')
ax.set_title(f'Coin Change: {amount}¢ = {used} ({n_coins} coins)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Interactive Experiment

In [ ]:
from ipywidgets import interact, IntSlider

@interact(n=IntSlider(value=10, min=3, max=20, description='Fib(n)'))
def overlap_visualizer(n):
    counts = {}
    def fib(k):
        counts[k] = counts.get(k, 0) + 1
        if k <= 1: return k
        return fib(k-1) + fib(k-2)
    counts.clear(); fib(n)
    
    ks = sorted(counts.keys())
    cs = [counts[k] for k in ks]
    clrs = ['#f43f5e' if c > 1 else '#10b981' for c in cs]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(ks, cs, color=clrs, alpha=0.8)
    ax.set_xlabel('Subproblem'); ax.set_ylabel('Calls')
    ax.set_title(f'fib({n}): {sum(cs)} total calls, only {len(ks)} unique subproblems', 
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()
    print(f'Savings with DP: {sum(cs) - len(ks)} redundant calls eliminated')

## 6. Tool Exploration

In [ ]:
from functools import lru_cache

# Python's built-in memoization decorator
@lru_cache(maxsize=None)
def fib_lru(n):
    if n <= 1: return n
    return fib_lru(n-1) + fib_lru(n-2)

print(f'fib(50) = {fib_lru(50)}')
print(f'Cache info: {fib_lru.cache_info()}')
print()
print('lru_cache gives you DP for free!')
print('hits = cache lookups, misses = actual computations')

## 7. Reflection Questions

1. Not all recursive problems benefit from DP. What two properties must a problem have?

2. Compare the recursion tree for fib(6) vs the DP table. How does memoization "collapse" the tree into a DAG?

3. In the coin change problem, why does the greedy approach (always pick largest coin) sometimes fail? Give an example with coins [1, 3, 4] and amount 6.

4. What's the space complexity of top-down vs bottom-up Fibonacci? Can bottom-up be optimized to O(1) space?

5. The 0/1 Knapsack problem has $n$ items and capacity $W$. The DP table is $n \times W$. Why is this "pseudo-polynomial" rather than truly polynomial?

---
*DP = recursion without the waste. → Exercises: `02_exercises.ipynb`*

## The Feynman Technique
Explain [this concept] in plain English to someone who has never seen it. Write it in the cell below. Then check: did you use any jargon you can't define from scratch? If yes, go back.

*(Write your explanation here...)*

## Review
- **Q:** 
- **A:** 

- **Q:** 
- **A:** 

## The Takeaway
> **The one thing to carry forward is:** *(Write the insight, not the formula)*